# 05 — Bootstrap Inference
Run 1000-iteration match-level bootstrap, compute 95% CIs, save bootstrap_results.json.

In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import os

from config import INTERIM_DIR, PROCESSED_DIR, N_BOOT, BOOT_SEED
from bootstrap import bootstrap_effects

In [2]:
# Load interim data
shots_df = pd.read_csv(os.path.join(INTERIM_DIR, 'shots.csv'))
team_windows_df = pd.read_csv(os.path.join(INTERIM_DIR, 'team_windows.csv'))
print(f'shots_df: {len(shots_df)} rows | team_windows_df: {len(team_windows_df)} rows')

shots_df: 1494 rows | team_windows_df: 10368 rows


In [3]:
# Run bootstrap (1000 iterations — may take several minutes)
print(f'Running {N_BOOT}-iteration bootstrap (seed={BOOT_SEED})...')
bootstrap_df = bootstrap_effects(team_windows_df, shots_df, n_boot=N_BOOT, seed=BOOT_SEED)
print(f'\nbootstrap_df shape: {bootstrap_df.shape}')
print(bootstrap_df.head())

Running 1000-iteration bootstrap (seed=42)...
  Bootstrap iteration 100/1000
  Bootstrap iteration 200/1000
  Bootstrap iteration 300/1000
  Bootstrap iteration 400/1000
  Bootstrap iteration 500/1000
  Bootstrap iteration 600/1000
  Bootstrap iteration 700/1000
  Bootstrap iteration 800/1000
  Bootstrap iteration 900/1000
  Bootstrap iteration 1000/1000
Saved bootstrap_results (1000 rows) → /Users/aidenpark/Project/cs109-soccer-momentum/data/processed/bootstrap_results.json

bootstrap_df shape: (1000, 7)
   iter  delta_shot_real  delta_goal_real  delta_shot_sim  delta_goal_sim  \
0     0         0.013924         0.021716        0.112442        0.111342   
1     1         0.038415         0.047895        0.208897        0.145458   
2     2         0.025511         0.037267        0.190578        0.124548   
3     3         0.023841         0.034869        0.155310        0.067863   
4     4         0.018278         0.051047        0.110307        0.087101   

   gap_shot  gap_goal  
0 

In [4]:
# 95% Confidence Intervals
print('=== 95% Confidence Intervals ===')
for col in ['delta_shot_real', 'delta_goal_real', 'delta_shot_sim', 'delta_goal_sim', 'gap_shot', 'gap_goal']:
    vals = bootstrap_df[col].dropna()
    lo, hi = np.percentile(vals, [2.5, 97.5])
    print(f'  {col:20s}: mean={vals.mean():.4f}  95% CI [{lo:.4f}, {hi:.4f}]')

=== 95% Confidence Intervals ===
  delta_shot_real     : mean=0.0308  95% CI [0.0047, 0.0600]
  delta_goal_real     : mean=0.0392  95% CI [0.0158, 0.0637]
  delta_shot_sim      : mean=0.1619  95% CI [0.0929, 0.2392]
  delta_goal_sim      : mean=0.1096  95% CI [0.0475, 0.1872]
  gap_shot            : mean=-0.1311  95% CI [-0.2062, -0.0621]
  gap_goal            : mean=-0.0704  95% CI [-0.1503, -0.0077]


In [5]:
# Check if gap > 0 (real exceeds null model)
gap_shot_ci_lo = np.percentile(bootstrap_df['gap_shot'].dropna(), 2.5)
gap_goal_ci_lo = np.percentile(bootstrap_df['gap_goal'].dropna(), 2.5)

print(f'\ngap_shot 2.5th percentile: {gap_shot_ci_lo:.4f}  (positive → real exceeds null)')
print(f'gap_goal 2.5th percentile: {gap_goal_ci_lo:.4f}  (positive → real exceeds null)')


gap_shot 2.5th percentile: -0.2062  (positive → real exceeds null)
gap_goal 2.5th percentile: -0.1503  (positive → real exceeds null)


In [6]:
# Verify file saved
out_path = os.path.join(PROCESSED_DIR, 'bootstrap_results.json')
print(f'bootstrap_results.json exists: {os.path.exists(out_path)}')
saved = pd.read_json(out_path)
print(f'Rows in saved file: {len(saved)}  [expect {N_BOOT}]')

bootstrap_results.json exists: True
Rows in saved file: 1000  [expect 1000]
